# Notebook 1 – Building the Knowledge Layer

This notebook walks through creating a **Foundry IQ** knowledge layer using Azure AI Search. By the end, you'll have:

1. A **search index** populated with NASA Earth at Night documents
2. A **SearchIndex knowledge source** backed by that index
3. A **Microsoft Learn MCP server knowledge source** (preview) that retrieves live documentation
4. A **knowledge base** that orchestrates agentic retrieval across both sources

The knowledge base combines indexed enterprise data with live web documentation, demonstrating how Foundry IQ unifies multiple content sources into a single retrieval endpoint.

## Architecture

```mermaid
flowchart LR
    subgraph KB[Knowledge Base]
        direction TB
        KS1["Knowledge Source 1: SearchIndex<br/>NASA Earth at Night docs<br/>Semantic + Vector search"]
        KS2["Knowledge Source 2: MCP Server<br/>Microsoft Learn<br/>microsoft_docs_search tool"]
    end

    Data[("Sample Data")] -->|push upload| KS1
    MCP["learn.microsoft.com/api/mcp"] -->|live query| KS2
    KB --> Retrieve["Agentic Retrieval Engine<br/><i>Query Planning &#8594; Parallel Search &#8594; Reranking</i>"]

    style KB fill:#f0f9ff,stroke:#0078d4,color:#000
    style Retrieve fill:#fffdf0,stroke:#d4a000,color:#000
```

## Prerequisites

- Azure AI Search service (Basic tier+) in an [agentic-retrieval-supported region](https://learn.microsoft.com/en-us/azure/search/search-region-support)
- Microsoft Foundry resource (AI Services endpoint: services.ai.azure.com) with an LLM deployment (e.g., `gpt-5.4-mini`) and an embedding model (`text-embedding-3-large`)
- System-assigned managed identity on the search service with **Cognitive Services User** on the Foundry resource
- User roles: Search Service Contributor, Search Index Data Contributor, Search Index Data Reader
- A `.env` file created from `.env.sample` with your resource details

## Phase 1A: Setup & Configuration

In [16]:
# Install required packages (uncomment if not already installed)
# %pip install azure-search-documents --pre azure-identity python-dotenv

In [17]:
import os
import json
from urllib.request import urlopen
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
)

load_dotenv()

# Azure AI Search
search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")

# Foundry project — derive AI Services endpoint from the project URL
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
ai_services_endpoint = project_endpoint.split("/api/projects")[0]

# Model deployments
llm_deployment = os.environ["LLM_DEPLOYMENT"]
llm_model = os.environ["LLM_MODEL"]
embedding_deployment = os.environ["EMBEDDING_DEPLOYMENT"]
embedding_model = os.environ["EMBEDDING_MODEL"]

# Use API key if provided, otherwise use DefaultAzureCredential
if search_api_key:
    search_credential = AzureKeyCredential(search_api_key)
else:
    search_credential = DefaultAzureCredential()

print(f"Search endpoint:      {search_endpoint}")
print(f"AI Services endpoint: {ai_services_endpoint}")
print(f"LLM:                  {llm_deployment} / {llm_model}")
print(f"Embedding:            {embedding_deployment} / {embedding_model}")
print(f"Auth method:          {'API Key' if search_api_key else 'DefaultAzureCredential'}")


Search endpoint:      https://search-agentic-memory-ds.search.windows.net
AI Services endpoint: https://npmsfoundrysc.services.ai.azure.com
LLM:                  gpt-5.4-mini / gpt-5.4-mini
Embedding:            text-embedding-3-large / text-embedding-3-large
Auth method:          DefaultAzureCredential


In [18]:
# Resource names used throughout this notebook
INDEX_NAME = "earth-at-night"
KNOWLEDGE_SOURCE_INDEX = "earth-at-night-ks"
KNOWLEDGE_SOURCE_MCP = "ms-learn-mcp-ks"
KNOWLEDGE_BASE_NAME = "foundry-iq-demo-kb"

## Phase 1B: Create Search Index & Ingest Data

We create an Azure AI Search index with:
- **Text fields** for document content and metadata
- **Vector field** for semantic similarity search (3072 dimensions for `text-embedding-3-large`)
- **Semantic configuration** for L2 reranking
- **Azure OpenAI vectorizer** so the retrieval engine can auto-vectorize queries

In [19]:
# Define the search index schema
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True,
        sortable=True,
        facetable=True,
    ),
    SearchableField(
        name="page_chunk",
        type=SearchFieldDataType.String,
        filterable=False,
        sortable=False,
        facetable=False,
    ),
    SearchField(
        name="page_embedding_text_3_large",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        vector_search_dimensions=3072,
        vector_search_profile_name="hnsw_text_3_large",
    ),
    SimpleField(
        name="page_number",
        type=SearchFieldDataType.Int32,
        filterable=True,
        sortable=True,
        facetable=True,
    ),
]

# Vectorizer: allows Azure AI Search to auto-vectorize text queries
vectorizer = AzureOpenAIVectorizer(
    vectorizer_name="azure_openai_text_3_large",
    parameters=AzureOpenAIVectorizerParameters(
        resource_url=ai_services_endpoint,
        deployment_name=embedding_deployment,
        model_name=embedding_model,
    ),
)

# Vector search configuration
vector_search = VectorSearch(
    profiles=[
        VectorSearchProfile(
            name="hnsw_text_3_large",
            algorithm_configuration_name="alg",
            vectorizer_name="azure_openai_text_3_large",
        )
    ],
    algorithms=[HnswAlgorithmConfiguration(name="alg")],
    vectorizers=[vectorizer],
)

# Semantic configuration for L2 reranking
semantic_config = SemanticConfiguration(
    name="semantic_config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="page_chunk")]
    ),
)

semantic_search = SemanticSearch(
    default_configuration_name="semantic_config",
    configurations=[semantic_config],
)

# Create the index
index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)

index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
result = index_client.create_or_update_index(index)
print(f"Index '{result.name}' created or updated successfully.")

Index 'earth-at-night' created or updated successfully.


### Upload Documents

We use the **NASA Earth at Night** sample data — pre-chunked JSON documents from [azure-search-sample-data](https://github.com/Azure-Samples/azure-search-sample-data). Each document contains a page chunk, its vector embedding, and page metadata.

In [20]:
# Download the sample documents
SAMPLE_DATA_URL = (
    "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/"
    "refs/heads/main/nasa-e-book/earth-at-night-json/documents.json"
)

with urlopen(SAMPLE_DATA_URL) as response:
    documents = json.loads(response.read().decode("utf-8"))

print(f"Downloaded {len(documents)} documents.")
print(f"Sample fields: {list(documents[0].keys())}")

Downloaded 194 documents.
Sample fields: ['id', 'page_chunk', 'page_embedding_text_3_large', 'page_number']


In [21]:
# Upload documents to the index (push-based ingestion)
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=INDEX_NAME,
    credential=search_credential,
)

# Upload in batches
BATCH_SIZE = 100
for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i : i + BATCH_SIZE]
    result = search_client.upload_documents(documents=batch)
    succeeded = sum(1 for r in result if r.succeeded)
    print(f"Batch {i // BATCH_SIZE + 1}: uploaded {succeeded}/{len(batch)} documents.")

print(f"\nAll documents uploaded to index '{INDEX_NAME}'.")

Batch 1: uploaded 100/100 documents.
Batch 2: uploaded 94/94 documents.

All documents uploaded to index 'earth-at-night'.


In [22]:
# Verify: run a simple search query
results = search_client.search(
    search_text="city lights at night",
    top=3,
    select=["id", "page_chunk", "page_number"],
)

print("Verification — top 3 results for 'city lights at night':")
for result in results:
    chunk_preview = result["page_chunk"][:150] + "..." if len(result["page_chunk"]) > 150 else result["page_chunk"]
    print(f"  [{result['id']}] (page {result['page_number']}): {chunk_preview}")

Verification — top 3 results for 'city lights at night':
  [earth_at_night_508_page_66_verbalized] (page 66): # Volcanoes

---

### Mount Etna Erupts - Italy

The highly active Mount Etna in Italy sent red lava rolling down its flank on March 19, 2017. An astr...
  [earth_at_night_508_page_39_verbalized] (page 39): # Figure: Night-Time View of the Surabaya Region

This figure is a night-time photograph showing a region including the city of Surabaya as well as va...
  [earth_at_night_508_page_35_verbalized] (page 35): # Figure: Nighttime View of Cairo and Surrounding Areas

**Date:** December 10, 2014

This figure presents a nighttime satellite image of the Cairo me...


## Phase 1C: Create SearchIndex Knowledge Source

A **knowledge source** is a reusable reference to content that a knowledge base queries. The `SearchIndexKnowledgeSource` wraps our existing index and specifies which fields to include in citation references.

> **sourceDataFields**: Controls which fields appear in the `references` array of retrieve responses. We include only human-readable fields (not embeddings).

In [23]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
)

# Create a knowledge source backed by the search index
index_knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_INDEX,
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="page_chunk"),
            SearchIndexFieldReference(name="page_number"),
        ],
    ),
)

index_client.create_or_update_knowledge_source(index_knowledge_source)
print(f"Knowledge source '{KNOWLEDGE_SOURCE_INDEX}' created successfully.")

Knowledge source 'earth-at-night-ks' created successfully.


## Phase 1D: Create MCP Server Knowledge Source (Preview)

An **MCP Server knowledge source** connects to any system that exposes a [Model Context Protocol](https://modelcontextprotocol.io/) endpoint. Unlike indexed sources, MCP sources query **live data at retrieval time** — no ingestion pipeline needed.

We connect to the public **Microsoft Learn MCP server** at `https://learn.microsoft.com/api/mcp`, which exposes a `microsoft_docs_search` tool that searches across all Microsoft documentation.

> **Note**: MCP server knowledge sources require `low` or `medium` retrieval reasoning effort. The `minimal` effort level is not supported.

In [24]:
from azure.search.documents.indexes.models import (
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerTool,
    McpServerAutoOutputParsing,
    McpServerToolInclusionMode,
)

# Create a knowledge source backed by the Microsoft Learn MCP server
mcp_knowledge_source = McpServerKnowledgeSource(
    name=KNOWLEDGE_SOURCE_MCP,
    description="Microsoft Learn documentation via MCP server. Use for questions about Azure services, SDKs, and cloud concepts.",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://learn.microsoft.com/api/mcp",
        tools=[
            McpServerTool(
                name="microsoft_docs_search",
                output_parsing=McpServerAutoOutputParsing(),
                inclusion_mode=McpServerToolInclusionMode.RERANKED,
                max_output_tokens=1000,
            )
        ],
    ),
)

index_client.create_or_update_knowledge_source(mcp_knowledge_source)
print(f"Knowledge source '{KNOWLEDGE_SOURCE_MCP}' created successfully.")

Knowledge source 'ms-learn-mcp-ks' created successfully.


In [25]:
# Verify: list all knowledge sources on the service
print("Knowledge sources on this service:")
for ks in index_client.list_knowledge_sources():
    print(f"  - {ks.name} ({type(ks).__name__})")

Knowledge sources on this service:
  - earth-at-night-ks (SearchIndexKnowledgeSource)
  - ms-learn-mcp-ks (McpServerKnowledgeSource)


## Phase 1E: Create Knowledge Base

A **knowledge base** orchestrates agentic retrieval over one or more knowledge sources. It defines:

- Which knowledge sources to query
- An LLM for query planning and answer synthesis
- Retrieval instructions that guide source selection
- Output mode (`answerSynthesis` for natural-language answers, or `extractedData` for raw grounding data)

Our knowledge base combines both sources and uses **retrieval instructions** to help the LLM route queries appropriately:
- Questions about satellite imagery, light pollution, and Earth observation → SearchIndex source
- Questions about Azure services, SDKs, and cloud documentation → Microsoft Learn MCP source
- Questions that span both domains → query both sources

In [26]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
    KnowledgeBaseAzureOpenAIModel,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalLowReasoningEffort,
)

# LLM configuration for query planning and answer synthesis
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=ai_services_endpoint,
    deployment_name=llm_deployment,
    model_name=llm_model,
)


# Create the knowledge base
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    knowledge_sources=[
        KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_INDEX),
        KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_MCP),
    ],
    description=(
        "A knowledge base combining NASA Earth at Night satellite data with "
        "live Microsoft Learn documentation for demonstration purposes."
    ),
    retrieval_instructions=(
        "Use the earth-at-night-ks knowledge source for questions about "
        "satellite imagery, light pollution, city lights, and Earth observation. "
        "Use the ms-learn-mcp-ks knowledge source for questions about Azure services, "
        "SDKs, cloud documentation, and Microsoft technologies. "
        "For questions that span both domains, query both sources."
    ),
    answer_instructions=(
        "Provide a clear, concise answer based on the retrieved documents. "
        "Cite sources using ref_id references. If information comes from "
        "multiple sources, synthesize them into a coherent response."
    ),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created successfully.")

Knowledge base 'foundry-iq-demo-kb' created successfully.


In [27]:
# Verify: list all knowledge bases on the service
print("Knowledge bases on this service:")
for kb in index_client.list_knowledge_bases():
    print(f"  - {kb.name}")

Knowledge bases on this service:
  - foundry-iq-demo-kb


## Phase 1F: Test the Knowledge Base

Now we test the knowledge base using the **retrieve action**. We'll send queries designed to exercise both knowledge sources:

1. **Cross-source query** — spans satellite data and Azure documentation
2. **Index-only query** — targets the NASA Earth at Night data
3. **MCP-only query** — targets Microsoft Learn
4. **Follow-up query** — demonstrates conversational context

The activity log in each response reveals which sources were queried and the subqueries generated by the LLM.

In [28]:
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseMessage,
    KnowledgeBaseRetrievalRequest,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient


def retrieve(kb_client, messages, reasoning_effort=None):
    """Send a retrieval request and display the response, activity, and references."""
    request = KnowledgeBaseRetrievalRequest()
    request.messages = []

    for msg in messages:
        if msg["role"] == "system":
            continue
        request.messages.append(
            KnowledgeBaseMessage(
                content=[KnowledgeBaseMessageTextContent(text=msg["content"])],
                role=msg["role"],
            )
        )

    request.retrieval_reasoning_effort = (
        reasoning_effort or KnowledgeRetrievalLowReasoningEffort()
    )
    request.include_activity = True

    result = kb_client.retrieve(request)

    # Display the synthesized answer
    answer = result.response[0].content[0].text
    print("=" * 80)
    print("ANSWER:")
    print("=" * 80)
    print(answer)

    # Display the activity log
    print("\n" + "-" * 80)
    print("ACTIVITY LOG:")
    print("-" * 80)
    for activity in result.activity:
        activity_type = type(activity).__name__
        print(f"  [{activity_type}] id={activity.id}, elapsed={activity.elapsed_ms}ms")
        if hasattr(activity, "knowledge_source_name"):
            print(f"    source: {activity.knowledge_source_name}")
        if hasattr(activity, "search_index_arguments") and activity.search_index_arguments:
            print(f"    search: {activity.search_index_arguments.search}")
        if hasattr(activity, "mcp_server_arguments") and activity.mcp_server_arguments:
            print(f"    tool: {activity.mcp_server_arguments.tool_name}")
            print(f"    args: {activity.mcp_server_arguments.tool_arguments}")
        if hasattr(activity, "input_tokens"):
            print(f"    tokens: in={activity.input_tokens}, out={activity.output_tokens}")

    # Display references
    print("\n" + "-" * 80)
    print(f"REFERENCES ({len(result.references)} total):")
    print("-" * 80)
    for ref in result.references[:5]:  # Show first 5
        ref_type = type(ref).__name__
        doc_key = getattr(ref, "doc_key", "N/A")
        score = getattr(ref, "reranker_score", None)
        score_str = f" (score: {score:.2f})" if score else ""
        print(f"  [{ref.id}] {ref_type}: {doc_key}{score_str}")

    return answer

In [29]:
# Create the retrieval client
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

print(f"Retrieval client ready for knowledge base: {KNOWLEDGE_BASE_NAME}")

Retrieval client ready for knowledge base: foundry-iq-demo-kb


In [30]:
# Query 1: Cross-source query — should hit both the indexed NASA data and MS Learn docs
messages = [
    {
        "role": "user",
        "content": (
            "What patterns of light pollution can be observed from satellite imagery, "
            "and what Azure AI Search features are available for building retrieval "
            "systems over scientific document collections?"
        ),
    }
]

print("Query 1: Cross-source (indexed data + Microsoft Learn MCP)")
print(f"Q: {messages[0]['content']}\n")
answer1 = retrieve(kb_client, messages)

Query 1: Cross-source (indexed data + Microsoft Learn MCP)
Q: What patterns of light pollution can be observed from satellite imagery, and what Azure AI Search features are available for building retrieval systems over scientific document collections?

ANSWER:
From the satellite-imagery sources, several clear light-pollution patterns are described: artificial night lights cluster most strongly around major metropolitan and industrial areas, especially in North America, Europe, and East Asia, while vast interiors such as the Amazon, much of Africa, deserts, and oceans remain comparatively dark.[ref_id:0][ref_id:1][ref_id:8] The sources also note that night lights often trace population density and urbanization, with strong illumination following coastlines, river corridors, and transportation/infrastructure networks.[ref_id:0][ref_id:11][ref_id:15] In change-over-time examples, brightness and spatial extent increase around growing cities and corridors, such as the Chicago region’s more 

In [31]:
# Query 2: Index-only — should primarily hit the NASA Earth at Night data
messages = [
    {
        "role": "user",
        "content": (
            "Why do suburban belts display larger December brightening than urban cores "
            "even though absolute light levels are higher downtown?"
        ),
    }
]

print("Query 2: Index-only (NASA Earth at Night data)")
print(f"Q: {messages[0]['content']}\n")
answer2 = retrieve(kb_client, messages)

Query 2: Index-only (NASA Earth at Night data)
Q: Why do suburban belts display larger December brightening than urban cores even though absolute light levels are higher downtown?

ANSWER:
Suburban belts show a larger December brightening because holiday lighting increases most dramatically in the suburbs and outskirts of major cities, where there is more yard space and a prevalence of single-family homes.[ref_id:14] Central urban areas also brighten during the holidays, but not as much; the source says they still experience a 20% to 30% increase, which is smaller than the suburban surge.[ref_id:14]

So even though downtown areas already have higher absolute light levels, the *relative* December increase is larger in suburban belts because there is more room and more home-based holiday decorating to add extra light there.[ref_id:14]

--------------------------------------------------------------------------------
ACTIVITY LOG:
-----------------------------------------------------------

In [32]:
# Query 3: MCP-only — should primarily hit the Microsoft Learn MCP server
messages = [
    {
        "role": "user",
        "content": "How do I create a knowledge base in Azure AI Search for agentic retrieval?",
    }
]

print("Query 3: MCP-only (Microsoft Learn documentation)")
print(f"Q: {messages[0]['content']}\n")
answer3 = retrieve(kb_client, messages)

Query 3: MCP-only (Microsoft Learn documentation)
Q: How do I create a knowledge base in Azure AI Search for agentic retrieval?

ANSWER:
To create a knowledge base for agentic retrieval in Azure AI Search, the portal quickstart says to go to **Agentic retrieval > Knowledge bases**, then choose **Add knowledge base**. [ref_id:0]

In the setup, you:
- enter **earth-at-night-kb** as the knowledge base name, [ref_id:0]
- add a **Chat completion model** by selecting **Add model deployment**, choosing **Microsoft Foundry**, then your **subscription**, **project**, and **LLM deployment**, [ref_id:0]
- set **Authentication type** to **System assigned identity**, [ref_id:0]
- select **earth-at-night-ks** under **Knowledge sources**, [ref_id:0]
- and then create the knowledge base. [ref_id:0]

The same quickstart notes that a knowledge base uses your knowledge source and deployed LLM to orchestrate agentic retrieval, where the LLM generates subqueries, Azure AI Search ranks the results semantica

In [33]:
# Query 4: Follow-up — demonstrates conversational context
messages = [
    {
        "role": "user",
        "content": (
            "Why do suburban belts display larger December brightening than urban cores "
            "even though absolute light levels are higher downtown?"
        ),
    },
    {
        "role": "assistant",
        "content": answer2,
    },
    {
        "role": "user",
        "content": "What about the Phoenix nighttime street grid — why is it so sharply visible from space?",
    },
]

print("Query 4: Follow-up (conversational context)")
print(f"Q: {messages[-1]['content']}\n")
answer4 = retrieve(kb_client, messages)

Query 4: Follow-up (conversational context)
Q: What about the Phoenix nighttime street grid — why is it so sharply visible from space?

ANSWER:
Phoenix is sharply visible from space because its metro area is laid out in a regular grid of city blocks and streets, and that street pattern is especially evident at night when the street lighting is visible from low Earth orbit.[ref_id:1] The city also has large brightly lit properties such as shopping centers, strip malls, gas stations, and major transport corridors like Grand Avenue, which make the grid stand out even more.[ref_id:1]

The broader metropolitan area includes many suburban and residential municipalities linked by surface streets and freeways, so the illuminated pattern spreads outward in a very organized way rather than staying confined to a dense downtown core.[ref_id:1] The source also notes that the grid follows the existing road network and creates a clear contrast with dark areas such as the Phoenix Mountains, agricultur

## Phase 1G: Cleanup

Uncomment and run the cells below to delete the resources created in this notebook.

> **Important**: If you plan to run Notebook 2 (Agent Integration), keep these resources — the agent will connect to this knowledge base.

In [34]:
# # Delete knowledge base
# index_client.delete_knowledge_base(KNOWLEDGE_BASE_NAME)
# print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' deleted.")

# # Delete knowledge sources
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE_INDEX)
# print(f"Knowledge source '{KNOWLEDGE_SOURCE_INDEX}' deleted.")

# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE_MCP)
# print(f"Knowledge source '{KNOWLEDGE_SOURCE_MCP}' deleted.")

# # Delete the search index
# index_client.delete_index(INDEX_NAME)
# print(f"Index '{INDEX_NAME}' deleted.")

## Summary

In this notebook, you:

1. **Created a search index** with text, vector, and semantic search capabilities
2. **Ingested** 200+ NASA Earth at Night document chunks using push-based indexing
3. **Created a SearchIndex knowledge source** that wraps the index for agentic retrieval
4. **Created an MCP Server knowledge source** (preview) that queries Microsoft Learn documentation at retrieval time
5. **Built a knowledge base** that combines both sources with LLM-powered query planning and answer synthesis
6. **Tested** the knowledge base with cross-source, single-source, and conversational queries

The activity logs in the responses show how the agentic retrieval engine decomposes queries into subqueries, routes them to the appropriate knowledge sources, and synthesizes unified answers.

### Next Steps

→ Continue to [Notebook 2: Agent Integration](foundry-agent-mcp-integration.ipynb) to connect this knowledge base to a Foundry Agent via MCP.